In [ ]:
from torchgeo.datasets import RasterDataset, stack_samples, VectorDataset,
from torchgeo.samplers import GridGeoSampler, RandomGeoSampler, Units

import torch

In [ ]:
# Загрузка данных ДЗЗ и маски в растровом виде (при необходимости использовать VectorDataset из torchgeo)

landsat_root = '...'
landsat = RasterDataset(paths=landsat_root)

mask_root = '...'
mask = RasterDataset(paths=mask_root)
mask.is_image = False 

In [ ]:
# Функция, при необходимости переклассифицирующая значения в маске — значение 255 можно использовать, если его нужно исключить из обучения
# Номера классов — с 0

def remap_mask(mask):
    remapped = 255 * torch.ones_like(mask, dtype=torch.long)
    remapped[mask == 0] = ...

    return remapped

In [ ]:
dataset = landsat & mask

In [ ]:
# Создание обучающей выборки — через семплеры

size = 512

train_sampler = RandomGeoSampler(dataset, size=size, length=350, units=Units.PIXELS)
val_sampler = RandomGeoSampler(dataset, size=size,  length=70, units=Units.PIXELS)
test_sampler = GridGeoSampler(dataset, size=size, stride=size, units=Units.PIXELS)

train_dataloader = DataLoader(dataset, batch_size=16, sampler=train_sampler, collate_fn=stack_samples, num_workers=4, persistent_workers=True)
val_dataloader = DataLoader(dataset, batch_size=16, sampler=val_sampler, collate_fn=stack_samples)
test_dataloader = DataLoader(dataset, batch_size=16, sampler=test_sampler, collate_fn=stack_samples)

In [ ]:
# Аугментации

# Функция, рассчитывающая показатели для нормализации
def calc_statistics(dset: RasterDataset):
    bounds = dset.bounds
    sample = dset[bounds]
    img = sample['image'].numpy().astype(np.float32)
    num_channels = img.shape[0]
    
    mean = img.reshape((num_channels, -1)).mean(axis=1)
    std  = img.reshape((num_channels, -1)).std(axis=1)

    return mean, std

mean, std = calc_statistics(landsat)

norm = K.Normalize(mean=mean, std=std)

# Аугментации для обучающей выборки
rotation = K.RandomRotation(degrees=25.0, resample='nearest', p=0.5)
rotation_90 = K.RandomRotation(degrees=90.0, resample='nearest', p=0.3)
crop = K.RandomCrop(size=(400, 400), pad_if_needed=True, padding_mode='reflect', p=0.2)
hflip = K.RandomHorizontalFlip(p=0.5)
vflip = K.RandomVerticalFlip(p=0.5) 
resize = K.Resize((512, 512), resample='nearest')


train_tfms = torch.nn.Sequential(rotation, rotation_90, crop, hflip, vflip, resize
)

val_tfms = torch.nn.Sequential(norm)